### Modelling Cash Use in South Africa

#### Step 0: Load libraries and set random seed

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import random
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, TabPanel, Tabs, LogScale
from bokeh.io import output_notebook
from dash import Dash, html, dcc, Output, Input
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
output_notebook() # Set up the notebook for Bokeh output

# Set random seed for reproducibility
np.random.seed(42)

#### Step 1: Generate a synthetic dataset

In [ ]:
# Define constants in the dataset for SA
PROVINCES = ['Western Cape', 'Gauteng', 'KwaZulu-Natal', 'Eastern Cape', 
            'Free State', 'Limpopo', 'Mpumalanga', 'North West', 'Northern Cape']

PROVINCE_POPULATIONS = {
    'Gauteng': 15830000,  # 25.1% of the SA population
    'KwaZulu-Natal': 12340000,  # 19.6%
    'Western Cape': 7400000,  # ~11.7%
    'Eastern Cape': 6800000,  # ~10.8%
    'Limpopo': 5800000,  # ~9.2%
    'Mpumalanga': 4800000,  # ~7.6%
    'North West': 4200000,  # ~6.7%
    'Free State': 2900000,  # ~4.6%
    'Northern Cape': 1360000,  # 2.2%
}

# Province income multipliers based on average income data
PROVINCE_INCOME_MULTIPLIERS = {
    'Western Cape': 1.75, #Income is highest in the Western Cape
    'Gauteng': 1.23,
    'Northern Cape': 0.88,
    'KwaZulu-Natal': 0.81,
    'Free State': 0.76,
    'Mpumalanga': 0.75,
    'Eastern Cape': 0.68,
    'North West': 0.69,
    'Limpopo': 0.63
}

# Population groups with proportions
POPULATION_GROUPS = {
    'Black African': 0.818,
    'Coloured': 0.082,
    'White': 0.073,
    'Indian/Asian': 0.027
}

# Generate sample of 10 000 representative individuals
def generate_individuals(num_individuals=10000):
    data = []
    
    for i in range(1, num_individuals + 1):
        # Generate demographic data
        age = max(18, int(np.random.normal(35, 12)))  # Adult population centered around 35 years
        gender = np.random.choice(['Male', 'Female', 'Other'], p=[0.484, 0.504, 0.012]) # Non-binary gender distribution
        
        # Weighted random province based on population
        province = np.random.choice(
            PROVINCES, 
            p=[PROVINCE_POPULATIONS[p]/sum(PROVINCE_POPULATIONS.values()) for p in PROVINCES]
        )
        
        # Weighted random population group
        population_group = np.random.choice(
            list(POPULATION_GROUPS.keys()),
            p=list(POPULATION_GROUPS.values())
        )
        
        # Generate income based on province and population group
        base_income = np.random.lognormal(mean=8.8, sigma=0.8)  # Centered around median income
        province_multiplier = PROVINCE_INCOME_MULTIPLIERS[province]
        
        # Adjustment for race-income inequality
        population_adjustments = {
            'White': 2.8,
            'Indian/Asian': 1.7,
            'Coloured': 0.9,
            'Black African': 0.7
        }
        
        population_multiplier = population_adjustments[population_group]
        
        # Apply multipliers and round off the figures
        monthly_income = round(base_income * province_multiplier * population_multiplier, -1)
        
        # Education level probabilities based on income (simplified)
        if monthly_income < 5000:
            education_probs = [0.15, 0.45, 0.35, 0.05]
        elif monthly_income < 15000:
            education_probs = [0.05, 0.25, 0.55, 0.15]
        else:
            education_probs = [0.01, 0.09, 0.45, 0.45]
            
        education_level = np.random.choice(
            ['No formal education', 'Primary', 'Secondary', 'Tertiary'],
            p=education_probs
        )
        
        # Employment status accounting for age and income level 
        if age >= 60:
            employment_probs = [0.1, 0.05, 0.05, 0.8, 0]
        elif monthly_income < 3000:
            employment_probs = [0.3, 0.5, 0.15, 0.01, 0.04]
        else:
            employment_probs = [0.75, 0.1, 0.1, 0.02, 0.03]
            
        employment_status = np.random.choice(
            ['Employed', 'Unemployed', 'Self-employed', 'Retired', 'Student'],
            p=employment_probs
        )
        
        # Main income source based on employment
        if employment_status == 'Employed':
            main_income_source = 'Salary'
        elif employment_status == 'Retired':
            main_income_source = np.random.choice(['Pension', 'Grants'], p=[0.7, 0.3])
        elif employment_status == 'Unemployed':
            main_income_source = np.random.choice(
                ['Grants', 'Remittances', 'Other'], 
                p=[0.7, 0.2, 0.1]
            )
        else:
            main_income_source = np.random.choice(
                ['Salary', 'Grants', 'Remittances', 'Other'],
                p=[0.5, 0.2, 0.2, 0.1]
            )
        
        # Banking behavior measured by regular use and not account ownership
        uses_bank_account = np.random.choice(['Yes', 'No'], p=[0.60, 0.40])
        
        # Cash withdrawal behavior
        if uses_bank_account == 'Yes':
            # 60% withdraw cash at least monthly
            withdrawal_freq_probs = [0.05, 0.35, 0.4, 0.2]
            
            # Socioeconomic factors of income and location influence withdrawal percentage
            if monthly_income < 5000 or province in ['Eastern Cape', 'Limpopo']:
                withdrawal_pct_mean = 80
                withdrawal_pct_std = 15
            elif monthly_income > 25000 and province in ['Western Cape', 'Gauteng']:
                withdrawal_pct_mean = 30
                withdrawal_pct_std = 15
            else:
                withdrawal_pct_mean = 55
                withdrawal_pct_std = 20
                
            preferred_channel_probs = [0.75, 0.05, 0.2]
            reason_probs = [0.37, 0.43, 0.1, 0.05, 0.05]
        else:
            withdrawal_freq_probs = [0, 0, 0, 1]
            withdrawal_pct_mean = 100
            withdrawal_pct_std = 0
            preferred_channel_probs = [0, 0, 0]
            reason_probs = [0, 1, 0, 0, 0]
            
        withdrawal_frequency = np.random.choice(
            ['Daily', 'Weekly', 'Monthly', 'Less often'],
            p=withdrawal_freq_probs
        )
        
        withdrawal_percentage = min(100, max(0, int(np.random.normal(withdrawal_pct_mean, withdrawal_pct_std))))
        
        if uses_bank_account == 'Yes':
            preferred_withdrawal_channel = np.random.choice(
                ['ATM', 'Bank branch', 'Retail store'],
                p=preferred_channel_probs
            )
            
            reason_for_cash_use = np.random.choice(
                ['Personal preference', 'Too little money', 'To avoid fees', 'Privacy', 'Other'],
                p=reason_probs
            )
        else:
            preferred_withdrawal_channel = 'None'
            reason_for_cash_use = 'No bank account'
        
        data.append({
            'id': i,
            'age': age,
            'gender': gender,
            'province': province,
            'population_group': population_group,
            'monthly_income': monthly_income,
            'education_level': education_level,
            'employment_status': employment_status,
            'main_income_source': main_income_source,
            'uses_bank_account': uses_bank_account,
            'withdrawal_frequency': withdrawal_frequency,
            'withdrawal_percentage': withdrawal_percentage,
            'preferred_withdrawal_channel': preferred_withdrawal_channel,
            'reason_for_cash_use': reason_for_cash_use
        })
    
    return pd.DataFrame(data)

# Generate the individuals dataset
individuals_df = generate_individuals(10000)

# Save the dataset to a CSV file
individuals_df.to_csv('C:/Users/36050/Desktop/.venv/cash_use_data/cash_use_sa_dataset.csv', index=False)
# individuals_df.to_csv('your/file/pathway/cash_use_sa_dataset.csv', index=False)

# Display the first few rows of the dataset
print(individuals_df.head())

#### Step 2: Generate summary statistics

In [10]:
# View the summary statistics of the dataset
print(individuals_df.describe(include='all'))

# Save summary statistics to a CSV file
summary = individuals_df.describe(include='all')
summary.to_csv('C:/Users/36050/Desktop/.venv/cash_use_data/summary_statistics.csv')

                 id          age  gender province population_group  \
count   10000.00000  10000.00000   10000    10000            10000   
unique          NaN          NaN       3        9                4   
top             NaN          NaN  Female  Gauteng    Black African   
freq            NaN          NaN    5023     2567             8167   
mean     5000.50000     35.10180     NaN      NaN              NaN   
std      2886.89568     11.18809     NaN      NaN              NaN   
min         1.00000     18.00000     NaN      NaN              NaN   
25%      2500.75000     26.00000     NaN      NaN              NaN   
50%      5000.50000     35.00000     NaN      NaN              NaN   
75%      7500.25000     43.00000     NaN      NaN              NaN   
max     10000.00000     82.00000     NaN      NaN              NaN   

        monthly_income education_level employment_status main_income_source  \
count     10000.000000           10000             10000              10000   
u

#### Step 3: Create a quick dashboard with *plotly express*

In [57]:
# Load data
df = pd.read_csv('C:/Users/36050/Desktop/.venv/cash_use_data/cash_use_sa_dataset.csv')

# --- Colors and font ---
colors = {
    'background': 'white',      # Dashboard background
    'header': 'black',          # Header text color
    'bar': '#339966',             # Bar color
    'pie': ['#339966', '#ff9933', '#3366cc', '#cc3366', '#ffcc00'],  # Pie chart colors
    'text': 'black'             # General text color
}
fonts = {
    'header': 'monospace',
    'body': 'monospace'
}

# --- Figures ---

# Average income by province (bar)
fig_bar = px.bar(
    df.groupby('province', as_index=False)['monthly_income'].mean(),
    x='province', y='monthly_income',
    title='Average income by province',
    color_discrete_sequence=[colors['bar']]
)
fig_bar.update_layout(
    plot_bgcolor=colors['background'],
    paper_bgcolor=colors['background'],
    font=dict(family=fonts['body'], color=colors['text'])
)

# Uses bank account ownership (pie)
fig_pie = px.pie(
    df, names='uses_bank_account',
    title='Bank account ownership',
    color_discrete_sequence=colors['pie']
)
fig_pie.update_layout(
    plot_bgcolor=colors['background'],
    paper_bgcolor=colors['background'],
    font=dict(family=fonts['body'], color=colors['text'])
)

# Withdrawl frequency (bar)
fig_cash_freq = px.bar(
    df['withdrawal_frequency'].value_counts().reset_index().sort_values('withdrawal_frequency', ascending=False),
    x='count', y='withdrawal_frequency',
    labels={'index': 'Cash Use Frequency', 'withdrawal_frequency': 'Count'},
    title='How often do people withdraw cash?',
    color_discrete_sequence=[colors['bar']]
)
fig_cash_freq.update_layout(
    plot_bgcolor=colors['background'],
    paper_bgcolor=colors['background'],
    font=dict(family=fonts['body'], color=colors['text'])
)

# Cash withdrawal methods (pie)
fig_withdrawal = px.pie(
    df, names='preferred_withdrawal_channel',
    title='Where do people prefer to withdraw cash from?',
    color_discrete_sequence=colors['pie']
)
fig_withdrawal.update_layout(
    plot_bgcolor=colors['background'],
    paper_bgcolor=colors['background'],
    font=dict(family=fonts['body'], color=colors['text'])
)

# Reasons for using cash (bar, top 10)
reason_counts = df['reason_for_cash_use'].value_counts().nlargest(10).reset_index()
fig_reasons = px.bar(
    reason_counts,
    x='count', y='reason_for_cash_use',
    labels={'index': 'Reason for Using Cash', 'reason_for_cash_use': 'Count'},
    title='Top Reasons for Using Cash',
    color_discrete_sequence=[colors['bar']]
)
fig_reasons.update_layout(
    plot_bgcolor=colors['background'],
    paper_bgcolor=colors['background'],
    font=dict(family=fonts['body'], color=colors['text'])
)

# --- App layout ---
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': colors['background'], 'fontFamily': fonts['body']},
    children=[
        html.H1(
            'South Africa Cash Use Dashboard',
            style={'color': colors['header'], 'fontFamily': fonts['header'], 'textAlign': 'center'}
        ),
        dcc.Graph(figure=fig_bar),
        dcc.Graph(figure=fig_pie),
        dcc.Graph(figure=fig_cash_freq),
        dcc.Graph(figure=fig_withdrawal),
        dcc.Graph(figure=fig_reasons)
    ]
)

if __name__ == '__main__':
    app.run(debug=True)

In [58]:
fig_bar.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/average_income_by_province.html")
fig_pie.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/bank_account_ownership.html")
fig_cash_freq.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/withdrawal_frequency.html")
fig_withdrawal.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/preferred_withdrawal_channel.html")
fig_reasons.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/reasons_for_cash_use.html")


#### Step 4: Create a mini app showing province, age, gender and frequency of cash use with *Plotly Express* and *ipywidgets* 

In [ ]:
# Create location and age widgets
province_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['province'].dropna().unique()),
    value='All',
    description='Province:'
)

age_slider = widgets.IntRangeSlider(
    value=[df['age'].min(), df['age'].max()],
    min=df['age'].min(),
    max=df['age'].max(),
    step=1,
    description='Age range:',
    continuous_update=False
)

# Define a global variable to store the figure
fig = None

def update_plot(province, age_range):
    global fig  # Declare the figure as 'global' to allow access outside the function
    filtered = df.copy()
    if province != 'All':
        filtered = filtered[filtered['province'] == province]
    filtered = filtered[(filtered['age'] >= age_range[0]) & (filtered['age'] <= age_range[1])]
    
    fig = px.histogram(
        filtered, 
        x='withdrawal_frequency', 
        color='gender',  # 'color' argument distinguishes categories by gender
        barmode='group',  # 'stack' can also be used instead of grouped bars 
        title='Withdrawal Frequency by Gender',
        labels={'withdrawal_frequency':'Withdrawal frequency'},
        color_discrete_sequence=px.colors.qualitative.Vivid # Set colour palette
    )
    fig.show()

ui = widgets.VBox([province_dropdown, age_slider])
out = widgets.interactive_output(update_plot, {'province': province_dropdown, 'age_range': age_slider})

display(ui, out)

Output()

In [ ]:
# Save current view as html
fig.write_html("C:/Users/36050/Desktop/.venv/cash_use_data/widget_view.html")

### Sources

https://www.resbank.co.za/content/dam/sarb/publications/monthly-releases/monthly-release-of-selected-data/2024/Eng%20423%20-%20May%202024.pdf
https://www.resbank.co.za/content/dam/sarb/publications/media-releases/2024/payments/SARB%20Payments%20Study%20Report%202023%20Executive%20Summary.pdf
https://www.stitch.money/blog/report-consumer-preference-south-africa-cash
https://finmark.org.za/Publications/FinScope_SA_Consumer_2023.pdf
https://www.stitch.money/blog/unlocking-the-potential-of-cash-at-atm-payments-in-south-africa
https://www.statssa.gov.za/?p=16738
https://www.statssa.gov.za/publications/P0302/MYPE%20Presentation%202024.pdf
https://www.statssa.gov.za/publications/P0100/IES_2023_Media_Media_Presentation_Final.pdf
https://www.resbank.co.za/content/dam/sarb/publications/reports/annual-reports/2024/SARB%20Annual%20Report%20202324.pdf
https://finmark.org.za/data-portal/ZAF?utf8=%E2%9C%93&data_topics%5B%5D=finscope-consumer&button=
https://www.statssa.gov.za/?p=17440
https://www.statssa.gov.za/publications/P0100/P01002022.pdf
https://www.resbank.co.za/en/home/what-we-do/statistics
https://finmark.org.za/data-portal/ZAF
https://www.statssa.gov.za/publications/P0318/GHS%202023%20Presentation.pdf
https://www.resbank.co.za/content/dam/sarb/publications/monetary-policy-review/2024/MPROCT2024INTERNET.pdf
https://www.finmark.org.za/knowledge-hub/articles/finscope-consumer-2023-south-africa-media-release?entity=news
https://www.statista.com/statistics/1467009/currency-in-circulation-in-south-africa/
https://www.fsca.co.za/Documents/FinScope%20SA%20Consumer%202021%20Survey%20%20Results%20Prepared%20for%20FSCA.pdf
https://tradingeconomics.com/south-africa/money-supply-m0
https://www.resbank.co.za/content/dam/sarb/publications/media-releases/2024/npsd---payments-report/Digital%20Payments%20Roadmap%20Report.pdf
https://www.statista.com/statistics/1428568/cash-adoption-in-south-africa/
https://www.ceicdata.com/en/south-africa/monetary-aggregates/money-supply-m3-sa
https://www.resbank.co.za/en/home/what-we-do/payments-and-settlements
https://www.sbv.co.za/wp-content/uploads/2024/03/SBV-Consumer-Cash-Survey-White-Paper.pdf
https://www.resbank.co.za/content/dam/sarb/publications/speeches/speeches-by-governors/2024/Opening%20address%20by%20Governor%20Lesetja%20Kganyago%20Governor%20of%20the%20South%20African%20Reserve%20Bank%20at%20the%202024%20Payments%20Conference.pdf
https://www.resbank.co.za/content/dam/sarb/publications/media-releases/2024/payments/SARB%20Payments%20Study%20Report%202023.pdf
https://www.cashmatters.org/blog/the-value-of-cash-and-payment-choice-in-south-africa
https://www.synthesis.co.za/is-there-a-real-desire-to-remove-cash-from-circulation-leading-banks-in-south-africa-weigh-in/
https://www.resbank.co.za/en/home/publications/publication-detail-pages/media-releases/2024/south-african-reserve-bank-payments-study-report-launch
https://www.statista.com/topics/12301/payment-methods-in-south-africa/
https://www.resbank.co.za/en/home/what-we-do/banknotes-and-coin
https://www.finmark.org.za/knowledge-hub/articles/finscope-msme-south-africa-2024-key-findings-highlight-urgent-need-for-informal-sector-support?entity=blog
https://www.statssa.gov.za/publications/P91194/P911942023.pdf
https://dailyinvestor.com/banking/74339/south-africans-dumping-cash-and-atms/
https://repository.up.ac.za/bitstream/handle/2263/59221/Wentzel_Investigation_2016.pdf
https://finmark.org.za/knowledge-hub/all?utf8=%3F&topics%5B%5D=finscope-consumer
https://dailyinvestor.com/banking/73630/big-changes-coming-to-atms-and-branches-at-south-africas-biggest-banks/
http://www.statssa.gov.za/questionnaires/QFS_P0044_Questionnaire.pdf
https://www.statssa.gov.za/publications/P91193/P911932023.pdf
https://www.statista.com/statistics/1350455/number-of-atm-in-south-africa-by-bank/
http://www.statssa.gov.za/questionnaires/AFS2022_P0021_Questionnaire.pdf
https://en.wikipedia.org/wiki/Demographics_of_South_Africa
https://www.statssa.gov.za/publications/P0100/IES_2023_Media_Media_Presentation_Final.pdf
https://ws.dws.gov.za/wsks/DefaultList.aspx?SubjectAreaID=1&DataTopicDetailID=1&DisplayTypeId=1&PerspectiveID=0&LvlID=10&DataTopicID=1
https://www.statista.com/statistics/1116077/total-population-of-south-africa-by-age-group/
https://www.statssa.gov.za/?p=15473
https://southafrica-info.com/land/nine-provinces-south-africa/
https://www.statssa.gov.za/publications/P0302/P03022024.pdf
https://www.statssa.gov.za/publications/P0100/P01002022.pdf
https://www.gov.za/news/media-statements/statistics-south-africa-census-2022-results-october-2023-22-aug-2024
https://www.statista.com/statistics/1330839/population-of-south-africa-by-age-group-and-gender/
https://www.statssa.gov.za/publications/P0318/P03182023.pdf
https://www.worldometers.info/demographics/south-africa-demographics/
https://www.parliament.gov.za/storage/app/media/Pages/2024/23-08-2024_NCOP_Three-sphere_Planning_Session/session1/Statistics_South_Africa_Patricia_Koka.pdf
https://x.com/StatsSA/status/1884157607901864346
https://www.wits.ac.za/news/latest-news/opinion/2023/2023-09/south-africa-cant-crack-the-inequality-curse-why-and-what-can-be-done.html
https://www.statssa.gov.za/?cat=23
https://documents1.worldbank.org/curated/en/099125003072240961/pdf/P1649270b73f1f0b5093fb0e644d33bc6f1.pdf
https://www.statssa.gov.za/publications/Report-03-10-19/Report-03-10-192017.pdf
https://data.worldbank.org/indicator/SI.POV.GINI?locations=ZA
https://www.statssa.gov.za/?p=15858
https://www.statista.com/outlook/co/socioeconomic-indicators/south-africa
https://www.resbank.co.za/content/dam/sarb/publications/media-releases/2024/payments/SARB%20Payments%20Study%20Report%202023%20Executive%20Summary.pdf
https://www.treasury.gov.za/comm_media/press/2023/2023112701%20An%20Inclusive%20Financial%20Sector%20for%20all%202023.pdf
https://www.statssa.gov.za/?p=16716
https://www.statssa.gov.za/?p=17430
https://en.wikipedia.org/wiki/List_of_South_African_provinces_by_population
https://www.statssa.gov.za/?p=17981
https://www.statssa.gov.za/publications/StatsInBrief/StatsInBrief2023.pdf
https://www.statssa.gov.za/?p=17283
https://www.statssa.gov.za/?p=12930
https://www.statssa.gov.za/publications/02-11-20/02-11-202022.pdf
https://dash.plotly.com/minimal-app